# Data Leakage: When a Model Learns From the Future

## One-hour machine learning case study

A college wants to predict, at the beginning of a semester, which students may fail a course. A model reports almost perfect accuracy.

That sounds impressive—but one variable in the dataset was created **after the semester ended**.

This notebook explores how data leakage creates unrealistic model performance and how to design a prediction pipeline that matches the real decision point.

> **Central question:** Did the model learn a useful pattern, or did it accidentally receive information from the future?

## Learning objectives

By the end of the session, students should be able to:

- define target leakage and train-test contamination;
- identify features that would not exist at prediction time;
- compare a leaked model with a valid model;
- explain why high accuracy can be evidence of a problem;
- build preprocessing inside a pipeline;
- separate training, validation, and test responsibilities;
- use AI to challenge a modeling plan rather than write it;
- communicate why a suspiciously strong model should not be deployed.

## Suggested schedule

| Time | Activity |
|---|---|
| 0–7 min | React to the near-perfect result |
| 7–16 min | Inspect the features and decision timeline |
| 16–28 min | Train the leaked model |
| 28–38 min | Remove leakage and rebuild correctly |
| 38–46 min | Investigate preprocessing leakage |
| 46–53 min | Use AI as a skeptical reviewer |
| 53–58 min | Transfer to a healthcare example |
| 58–60 min | Exit reflection |

# Part 1 — The Claim

The analytics team reports:

> “Our model predicts course failure with more than 98% accuracy.”

Before seeing the data, answer:

1. What questions would you ask about the evaluation?
2. Could extremely high performance be suspicious?
3. What information would not be available at the beginning of the semester?
4. What would happen if future information entered the model?

### Initial response

**Questions about the evaluation:**  

**Potentially unavailable variables:**  

**Why very high accuracy may be suspicious:**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn.impute import SimpleImputer

rng = np.random.default_rng(42)
n = 1200

prior_gpa = np.clip(rng.normal(2.7, 0.65, n), 0, 4)
attendance_first_2_weeks = np.clip(rng.normal(0.86, 0.12, n), 0, 1)
placement_score = np.clip(rng.normal(70, 14, n), 20, 100)
work_hours = np.clip(rng.normal(22, 12, n), 0, 60)

risk_logit = (
    -2.2
    - 1.1 * (prior_gpa - 2.5)
    - 2.2 * (attendance_first_2_weeks - 0.75)
    - 0.025 * (placement_score - 65)
    + 0.025 * (work_hours - 20)
)
failure_prob = 1 / (1 + np.exp(-risk_logit))
failed = rng.binomial(1, failure_prob)

# Leakage: recorded after the course is essentially complete.
final_exam_score = np.clip(
    82 - 35 * failed + rng.normal(0, 8, n),
    0, 100
)
final_letter_grade_numeric = np.clip(
    4 - 3.4 * failed + rng.normal(0, 0.35, n),
    0, 4
)

df = pd.DataFrame({
    "prior_gpa": prior_gpa,
    "attendance_first_2_weeks": attendance_first_2_weeks,
    "placement_score": placement_score,
    "work_hours": work_hours,
    "final_exam_score": final_exam_score,
    "final_letter_grade_numeric": final_letter_grade_numeric,
    "failed_course": failed
})

df.head()

# Part 2 — Build the Decision Timeline

Assume the prediction must be made at the end of the second week.

Classify each feature:

| Feature | Available by week 2? | Safe to use? | Reason |
|---|---|---|---|
| Prior GPA | | | |
| First-two-week attendance | | | |
| Placement score | | | |
| Weekly work hours | | | |
| Final exam score | | | |
| Final course grade | | | |

A feature is not valid merely because it appears in the database. It must exist at the moment the prediction will be made.

In [ ]:
df.groupby("failed_course").mean(numeric_only=True).round(2)

# Part 3 — Train the Leaked Model

First, train a model using every available column except the target.

In [ ]:
X_leaked = df.drop(columns="failed_course")
y = df["failed_course"]

X_train, X_test, y_train, y_test = train_test_split(
    X_leaked, y, test_size=0.25, random_state=42, stratify=y
)

leaked_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])

leaked_model.fit(X_train, y_train)
pred_leaked = leaked_model.predict(X_test)

leaked_metrics = {
    "Accuracy": accuracy_score(y_test, pred_leaked),
    "Precision": precision_score(y_test, pred_leaked),
    "Recall": recall_score(y_test, pred_leaked),
}
pd.Series(leaked_metrics).to_frame("Leaked model").style.format("{:.1%}")

In [ ]:
pd.DataFrame(
    confusion_matrix(y_test, pred_leaked),
    index=["Actual pass", "Actual fail"],
    columns=["Predicted pass", "Predicted fail"]
)

## Interpretation

1. Is the performance believable for a week-two prediction?
2. Which variables are doing most of the work?
3. Would this model perform the same way in production?
4. Why is this result not a real achievement?

In [ ]:
feature_names = X_leaked.columns
coefficients = leaked_model.named_steps["model"].coef_[0]

importance = pd.Series(
    np.abs(coefficients),
    index=feature_names
).sort_values(ascending=False)

importance.to_frame("Absolute standardized coefficient")

# Part 4 — Remove the Future

Rebuild the model using only information that exists by week two.

In [ ]:
valid_features = [
    "prior_gpa",
    "attendance_first_2_weeks",
    "placement_score",
    "work_hours"
]

X_valid = df[valid_features]

X_train, X_test, y_train, y_test = train_test_split(
    X_valid, y, test_size=0.25, random_state=42, stratify=y
)

valid_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])

valid_model.fit(X_train, y_train)
pred_valid = valid_model.predict(X_test)

valid_metrics = {
    "Accuracy": accuracy_score(y_test, pred_valid),
    "Precision": precision_score(y_test, pred_valid, zero_division=0),
    "Recall": recall_score(y_test, pred_valid, zero_division=0),
}

comparison = pd.DataFrame({
    "Leaked model": leaked_metrics,
    "Valid week-two model": valid_metrics
})

comparison.style.format("{:.1%}")

## Critical discussion

The valid model performs worse, but it is more honest.

Explain:

- Why lower performance may represent better science.
- Why the valid model is more useful for a real intervention.
- What information the team should collect to improve the model legitimately.

In [ ]:
cv_scores = cross_val_score(
    valid_model,
    X_valid,
    y,
    scoring="accuracy",
    cv=5
)

pd.Series(cv_scores, name="Cross-validation accuracy").to_frame().style.format("{:.1%}")

# Part 5 — Preprocessing Leakage

Leakage can also happen when preprocessing is performed before the train-test split.

Examples include:

- imputing missing values using the full dataset;
- scaling using the full dataset;
- selecting features using all labels;
- oversampling before splitting;
- tuning hyperparameters on the test set.

The safest practice is to place preprocessing steps inside a machine-learning pipeline so that each step is learned only from the training data.

## AI as a skeptical reviewer

After writing your own pipeline plan, use one prompt:

> Review this modeling plan for possible data leakage. Ask questions rather than rewriting the solution.

> Which variables in this college dataset might be created after the prediction date?

> Explain three ways preprocessing can contaminate the test set.

> What evidence would convince you that the evaluation matches production conditions?

Evaluate the response:

**Useful warning:**  

**Claim requiring verification:**  

**Leakage risk the AI missed:**  

**Change I will make:**

# Transfer task — Healthcare

A hospital wants to predict patient readmission at discharge.

A model includes:

- age;
- diagnoses recorded at discharge;
- medication list at discharge;
- number of follow-up appointments completed in the next 30 days;
- whether the patient was readmitted within 30 days.

Identify:

1. the target;
2. the likely leaked feature;
3. the exact prediction moment;
4. a safer feature set;
5. one preprocessing step that must be learned from training data only.

# Exit reflection

Complete the statements:

- Before this lesson, I thought high accuracy meant …
- The clearest sign of leakage was …
- A prediction feature is valid only when …
- A pipeline helps because …
- The model I would trust more is …

# Instructor checklist

- [ ] Students built a decision timeline.
- [ ] Students identified target leakage.
- [ ] Students compared suspicious and realistic performance.
- [ ] Students understood that lower performance can be more valid.
- [ ] Preprocessing was placed inside a pipeline.
- [ ] AI was used to audit reasoning, not build the final answer.
- [ ] Students transferred the concept to another domain.

# Closing principle

A model should never be evaluated with information it will not possess when it must make a real prediction.

The goal is not to produce the highest score. It is to produce the most honest estimate of future performance.